# Data Quality Report
## Sentiment Analysis Dataset — DataQualityAgent

| | |
|---|---|
| **Dataset** | `data/raw/unified_dataset.csv` |
| **ML task** | Binary sentiment classification (positive / negative) |
| **Sources** | IMDB reviews · books.toscrape.com · OpenLibrary API |

**Structure:**
- **Part 1 — Detective**: detect and visualize all quality issues
- **Part 2 — Surgeon**: apply 2 cleaning strategies, compare results
- **Part 3 — Argument**: justify the best strategy
- **Bonus — LLM**: Claude API explains issues and recommends strategy

In [ ]:
import sys, pathlib
ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, Markdown

from agents.data_quality_agent import DataQualityAgent

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

CLEAN_DIR = ROOT / 'data' / 'clean'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = ROOT / 'data' / 'raw' / 'unified_dataset.csv'
assert RAW_PATH.exists(), f'Run python run_agent.py first! ({RAW_PATH})'

df_raw = pd.read_csv(RAW_PATH)
print(f'Loaded {len(df_raw):,} rows  |  columns: {list(df_raw.columns)}')
df_raw.head(3)

## Preparing a realistic dirty dataset

Real-world pipelines accumulate noise. We inject synthetic defects into a copy of the
dataset to demonstrate detection and repair capabilities:

| Injected defect | Quantity | Rationale |
|---|---|---|
| Missing `text` values | 5% of rows | Simulates upstream pipeline failures |
| Duplicate rows | 4% of rows | Simulates repeated API calls / crawls |
| Class imbalance | 2× ratio | Simulates biased scraping |

Outliers (very short book titles vs very long IMDB reviews) are **already present**
naturally in the mixed-source dataset.

In [ ]:
rng = np.random.default_rng(seed=99)
df_dirty = df_raw.copy()

# 1. Inject missing values (5%)
n_missing = int(len(df_dirty) * 0.05)
missing_idx = rng.choice(len(df_dirty), n_missing, replace=False)
df_dirty.loc[missing_idx, 'text'] = np.nan

# 2. Inject duplicate rows (4%)
n_dups = int(len(df_dirty) * 0.04)
dup_idx = rng.choice(len(df_dirty), n_dups, replace=False)
df_dirty = pd.concat([df_dirty, df_dirty.iloc[dup_idx]], ignore_index=True)

# 3. Skew class balance: drop 60% of 'positive' rows to create 2× imbalance
pos_idx = df_dirty[df_dirty['label'] == 'positive'].index
drop_pos = rng.choice(pos_idx, int(len(pos_idx) * 0.60), replace=False)
df_dirty = df_dirty.drop(index=drop_pos).reset_index(drop=True)

# 4. Outliers already present — just report them
print('Dirty dataset shape:', df_dirty.shape)
print(df_dirty['label'].value_counts().to_string())

---
# Part 1 — Detective 🔍
Detect all quality issues and visualize each one.

In [ ]:
agent = DataQualityAgent(
    label_col='label',
    text_col='text',
    outlier_method='iqr',
    iqr_factor=1.5,
    imbalance_threshold=1.5,
)

report = agent.detect_issues(df_dirty)
print(report.summary())

### 1.1 Missing values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart — missing count per column
miss_counts = pd.Series(
    {col: int(df_dirty[col].isna().sum()) for col in df_dirty.columns}
)
miss_counts_nonzero = miss_counts[miss_counts > 0]
if miss_counts_nonzero.empty:
    axes[0].text(0.5, 0.5, 'No missing values', ha='center', va='center',
                 transform=axes[0].transAxes, fontsize=14)
else:
    miss_counts_nonzero.plot.barh(ax=axes[0], color='#EF5350')
    axes[0].set_title('Missing values per column', fontweight='bold')
    axes[0].set_xlabel('Count')

# Heat-map style: NaN presence per row sample
sample_size = min(300, len(df_dirty))
sample_df = df_dirty.sample(sample_size, random_state=0)
nan_matrix = sample_df.isna().astype(int)
sns.heatmap(nan_matrix.T, ax=axes[1], cmap=['#E8F5E9','#EF5350'],
            cbar=False, linewidths=0)
axes[1].set_title(f'NaN heatmap (random {sample_size} rows)', fontweight='bold')
axes[1].set_xlabel('Row index (sample)')
axes[1].set_ylabel('Column')

plt.tight_layout()
plt.savefig(ROOT / 'data/raw/qplot_missing.png', bbox_inches='tight')
plt.show()

print(f'Missing text rows: {report.missing.total_affected_rows}  '
      f'({report.missing.total_affected_rows/len(df_dirty)*100:.1f}%)')

### 1.2 Duplicate rows

In [ ]:
dup_mask = df_dirty.duplicated(keep='first')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Pie — unique vs duplicate
sizes = [len(df_dirty) - dup_mask.sum(), dup_mask.sum()]
axes[0].pie(sizes, labels=['Unique', 'Duplicate'],
            autopct='%1.1f%%', colors=['#66BB6A', '#EF5350'],
            startangle=90, wedgeprops={'edgecolor': 'white'})
axes[0].set_title('Unique vs Duplicate rows', fontweight='bold')

# Duplicates per source
dup_by_src = df_dirty[dup_mask]['source'].value_counts()
if not dup_by_src.empty:
    dup_by_src.plot.bar(ax=axes[1], color='#EF5350', edgecolor='white')
    axes[1].set_title('Duplicate rows per source', fontweight='bold')
    axes[1].set_xlabel('')
    axes[1].set_ylabel('Count')
    axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(ROOT / 'data/raw/qplot_duplicates.png', bbox_inches='tight')
plt.show()

print(f'Duplicates: {report.duplicates.count} rows ({report.duplicates.percentage:.1f}%)')
# Show a few duplicate pairs
if report.duplicates.sample_indices:
    print('\nSample duplicate rows:')
    display(df_dirty.iloc[report.duplicates.sample_indices[:3]][['text', 'label', 'source']])

### 1.3 Text-length outliers

Outliers are identified by word count using the **IQR fences** method:
- Lower fence: Q1 − 1.5 × IQR  
- Upper fence: Q3 + 1.5 × IQR  

> **Key insight:** Our dataset mixes long IMDB reviews (~230 words avg) with short book
> titles (~4 words avg). This cross-source variance naturally produces many outliers
> when detection is run on the whole dataset.

In [ ]:
wc = df_dirty['text'].dropna().str.split().str.len()
q1, q3 = wc.quantile(0.25), wc.quantile(0.75)
iqr = q3 - q1
lower = max(0, q1 - 1.5 * iqr)
upper = q3 + 1.5 * iqr

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Box plot per source
src_wc = df_dirty[['source']].copy()
src_wc['word_count'] = df_dirty['text'].str.split().str.len()
src_wc_clean = src_wc.dropna()
order = src_wc_clean.groupby('source')['word_count'].median().sort_values().index.tolist()
sns.boxplot(data=src_wc_clean, x='source', y='word_count',
            order=order, palette='muted', ax=axes[0],
            flierprops={'marker': '.', 'alpha': 0.3})
axes[0].set_title('Word count per source', fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)

# Histogram with IQR fences
wc_clipped = wc.clip(upper=wc.quantile(0.99))
axes[1].hist(wc_clipped, bins=60, color='#42A5F5', edgecolor='none', alpha=0.8)
axes[1].axvline(lower, color='#FF7043', lw=2, ls='--', label=f'Lower fence {lower:.0f}')
axes[1].axvline(upper, color='#FF7043', lw=2, ls='--', label=f'Upper fence {upper:.0f}')
axes[1].set_title('Word count distribution + IQR fences', fontweight='bold')
axes[1].set_xlabel('Words')
axes[1].legend(fontsize=9)

# Scatter: outliers vs inliers coloured
wc_all = df_dirty['text'].str.split().str.len().fillna(0)
is_outlier = (wc_all < lower) | (wc_all > upper)
color_map = is_outlier.map({True: '#EF5350', False: '#66BB6A'})
axes[2].scatter(range(len(wc_all)), wc_all.clip(upper=wc.quantile(0.995)),
                c=color_map, s=2, alpha=0.4)
axes[2].axhline(lower, color='#FF7043', lw=1.5, ls='--')
axes[2].axhline(upper, color='#FF7043', lw=1.5, ls='--')
red_p = mpatches.Patch(color='#EF5350', label=f'Outlier ({is_outlier.sum()})')
grn_p = mpatches.Patch(color='#66BB6A', label=f'Inlier ({(~is_outlier).sum()})')
axes[2].legend(handles=[red_p, grn_p], fontsize=9)
axes[2].set_title('Row-level outlier map', fontweight='bold')
axes[2].set_xlabel('Row index')
axes[2].set_ylabel('Word count')

plt.tight_layout()
plt.savefig(ROOT / 'data/raw/qplot_outliers.png', bbox_inches='tight')
plt.show()

print(f'Outliers: {report.outliers.count} rows ({report.outliers.percentage:.1f}%)')
print(f'IQR bounds: [{lower:.0f}, {upper:.0f}] words')

### 1.4 Class imbalance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
lbl_palette = {'positive': '#66BB6A', 'negative': '#EF5350'}

# Overall balance
lbl_counts = df_dirty['label'].value_counts()
colors = [lbl_palette.get(l, '#90A4AE') for l in lbl_counts.index]
bars = axes[0].bar(lbl_counts.index, lbl_counts.values, color=colors, edgecolor='white')
for bar, val in zip(bars, lbl_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(val), ha='center', fontsize=11)
axes[0].set_title('Overall label balance', fontweight='bold')
axes[0].set_ylabel('Count')

# Per-source balance
pivot = df_dirty.groupby(['source', 'label']).size().unstack(fill_value=0)
pivot.plot(kind='bar', ax=axes[1],
           color=[lbl_palette.get(c, '#90A4AE') for c in pivot.columns],
           edgecolor='white', rot=25)
axes[1].set_title('Label balance per source', fontweight='bold')
axes[1].legend(title='Label')
axes[1].set_xlabel('')

# Imbalance ratio per source
ratios = {}
for src in df_dirty['source'].unique():
    sub = df_dirty[df_dirty['source'] == src]['label'].value_counts()
    if len(sub) >= 2:
        ratios[src] = sub.max() / sub.min()
    else:
        ratios[src] = float('inf')
ratio_s = pd.Series(ratios).sort_values(ascending=False)
colors2 = ['#EF5350' if r > 1.5 else '#66BB6A' for r in ratio_s.values]
axes[2].bar(ratio_s.index, ratio_s.values, color=colors2, edgecolor='white')
axes[2].axhline(1.5, color='gray', ls='--', lw=1.5, label='threshold 1.5×')
axes[2].set_title('Imbalance ratio per source', fontweight='bold')
axes[2].set_ylabel('majority / minority')
axes[2].tick_params(axis='x', rotation=20)
axes[2].legend()

plt.tight_layout()
plt.savefig(ROOT / 'data/raw/qplot_imbalance.png', bbox_inches='tight')
plt.show()

print(f'Imbalance ratio: {report.imbalance.imbalance_ratio:.2f}×')
print('Class counts:', report.imbalance.class_counts)
print('Is imbalanced:', report.imbalance.is_imbalanced)

---
# Part 2 — Surgeon 🔧
Apply two different cleaning strategies and compare the results.

| | Strategy A — Aggressive | Strategy B — Conservative |
|---|---|---|
| **missing** | `drop` — remove rows | `fill` — replace with placeholder |
| **duplicates** | `drop` — keep first occurrence | `drop` — same |
| **outliers** | `drop_iqr` — remove outlier rows | `clip_iqr` — truncate long texts |
| **imbalance** | `undersample` — balance classes | `oversample` — duplicate minority |

In [ ]:
print('=== Strategy A: Aggressive ===')
strategy_a = {
    'missing': 'drop',
    'duplicates': 'drop',
    'outliers': 'drop_iqr',
    'imbalance': 'undersample',
}
df_clean_a = agent.fix(df_dirty, strategy=strategy_a)
df_clean_a.to_csv(CLEAN_DIR / 'cleaned_strategy_a.csv', index=False)
print(f'\nResult: {len(df_clean_a):,} rows')
print(df_clean_a['label'].value_counts().to_string())

In [ ]:
print('=== Strategy B: Conservative ===')
strategy_b = {
    'missing': 'fill',
    'duplicates': 'drop',
    'outliers': 'clip_iqr',
    'imbalance': 'oversample',
}
df_clean_b = agent.fix(df_dirty, strategy=strategy_b)
df_clean_b.to_csv(CLEAN_DIR / 'cleaned_strategy_b.csv', index=False)
print(f'\nResult: {len(df_clean_b):,} rows')
print(df_clean_b['label'].value_counts().to_string())

### Comparison table: dirty vs Strategy A vs Strategy B

In [ ]:
cmp_a = agent.compare(df_dirty, df_clean_a, strategy_label='A: Aggressive')
cmp_b = agent.compare(df_dirty, df_clean_b, strategy_label='B: Conservative')

comparison = pd.concat([cmp_a, cmp_b], ignore_index=True)
print('\n=== Before / After comparison ===')

def _color_change(v):
    if isinstance(v, str) and v.startswith("-"):
        return "color: #1B5E20"
    if isinstance(v, str) and v.startswith("+"):
        return "color: #B71C1C"
    return ""

display(comparison.style
    .map(_color_change, subset=["change_pct"])
    .set_caption('← negative change = improvement for most metrics')
    .format(precision=2))


In [ ]:
metrics_to_plot = ['total_rows', 'missing_rows', 'duplicate_rows', 'outlier_rows', 'imbalance_ratio']

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(18, 5))

for ax, metric in zip(axes, metrics_to_plot):
    dirty_val  = float(cmp_a[cmp_a['metric'] == metric]['before'].values[0])
    clean_a_val = float(cmp_a[cmp_a['metric'] == metric]['after'].values[0])
    clean_b_val = float(cmp_b[cmp_b['metric'] == metric]['after'].values[0])
    
    vals  = [dirty_val, clean_a_val, clean_b_val]
    labels = ['Dirty', 'Strategy A', 'Strategy B']
    colors = ['#EF5350', '#42A5F5', '#66BB6A']
    
    bars = ax.bar(labels, vals, color=colors, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{v:.1f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(metric.replace('_', ' ').title(), fontweight='bold', fontsize=10)
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Quality metrics: Dirty → Strategy A → Strategy B', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'data/raw/qplot_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
datasets = [
    (df_dirty,   'Dirty',       '#EF5350'),
    (df_clean_a, 'Strategy A',  '#42A5F5'),
    (df_clean_b, 'Strategy B',  '#66BB6A'),
]
for ax, (df, title, color) in zip(axes, datasets):
    wc = df['text'].dropna().str.split().str.len()
    wc.clip(upper=wc.quantile(0.99)).plot.hist(bins=50, ax=ax, color=color, alpha=0.8)
    ax.axvline(wc.median(), color='black', ls='--', lw=1.5, label=f'median={wc.median():.0f}')
    ax.set_title(f'Word count — {title}', fontweight='bold')
    ax.set_xlabel('Words')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(ROOT / 'data/raw/qplot_wordcount_compare.png', bbox_inches='tight')
plt.show()

---
# Part 3 — Argument 🧠
## Justification for the recommended cleaning strategy

### Context
We are building a **binary sentiment classifier** (positive / negative) on a mixed-source
text dataset containing long IMDB movie reviews (~230 words), medium-length OpenLibrary
titles (~6 words), and short book titles from scraping (~5 words).

---

### Issue 1 — Missing values → **`drop`** ✅
Missing text means there is literally nothing to classify. Both strategies remove or
fill these rows, but filling with `[MISSING]` creates a spurious token that the model
would learn as a feature — potentially causing "[MISSING] → negative" associations.
**Dropping** is safer because it preserves distributional integrity of the training set.
With only ~5% affected rows the information loss is negligible.

---

### Issue 2 — Duplicates → **`drop`** ✅
Duplicate rows in the training set cause a model to implicitly up-weight those specific
examples. For sentiment analysis this could lead to overfitting on repeated reviews and
inflated evaluation metrics (if the same row appears in both train and validation splits).
Dropping duplicates (keeping first occurrence) is the correct choice in both strategies.

---

### Issue 3 — Text-length outliers → **`clip_iqr`** (Strategy B) ✅
The outliers here arise from source heterogeneity rather than data corruption:
- IMDB reviews are legitimately long (useful content throughout).
- Book titles are legitimately short (not noise).

**`drop_iqr`** (Strategy A) removes ~40% of all rows including valid IMDB reviews,
leading to massive information loss.  
**`clip_iqr`** (Strategy B) truncates only the excessively long tail, preserving all
rows while capping inputs at a reasonable length for transformer-based models
(BERT has a 512-token limit anyway). This is the better choice for NLP.

---

### Issue 4 — Class imbalance → **`undersample`** (Strategy A) ✅
The injected 2× imbalance would bias a classifier towards predicting "negative" more
often.  
- **`oversample`** (Strategy B) creates synthetic duplicates by repeating minority-class
  examples, which can cause memorisation rather than generalisation.
- **`undersample`** reduces the majority class, discarding some data but producing
  a perfectly balanced training set with no duplicates — cleaner for evaluation.

For small datasets (< 5k), undersampling is preferred over oversampling because the
latter inflates the dataset with exact copies.

---

### ✅ Recommended final strategy
```python
strategy = {
    'missing':    'drop',         # preserves distributional integrity
    'duplicates': 'drop',         # prevents train-val leakage
    'outliers':   'clip_iqr',     # retains all rows, handles transformer limits
    'imbalance':  'undersample',  # clean balance without synthetic duplicates
}
```
This hybrid (B for outliers, A for everything else) maximises data quality while
minimising information loss — a key constraint in small mixed-source NLP datasets.

In [ ]:
# Apply the recommended hybrid strategy and save as the canonical clean dataset
best_strategy = {
    'missing': 'drop',
    'duplicates': 'drop',
    'outliers': 'clip_iqr',
    'imbalance': 'undersample',
}
df_best = agent.fix(df_dirty, strategy=best_strategy)
df_best.to_csv(CLEAN_DIR / 'cleaned_best.csv', index=False)
print(f'Best strategy result: {len(df_best):,} rows')
print(df_best['label'].value_counts().to_string())

---
# Bonus — LLM Explanation 🤖

The `explain_with_llm` skill sends the QualityReport to **Claude** via the Anthropic API.
Claude explains each issue in the context of our ML task and recommends a cleaning strategy.

Requires `ANTHROPIC_API_KEY` in `.env` at the project root.

In [ ]:
llm_response = agent.explain_with_llm(
    report=report,
    task_description="binary sentiment classification (positive / negative) "
                     "on a mixed-source text dataset (movie reviews + book titles)",
)
display(Markdown(llm_response))

---
## Summary

In [ ]:
print('=== FINAL SUMMARY ===')
for name, df_s in [('Dirty', df_dirty), ('Strategy A', df_clean_a),
                   ('Strategy B', df_clean_b), ('Best (hybrid)', df_best)]:
    r = agent.detect_issues(df_s)
    print(f'\n  {name}:')
    print(f'    rows={r.total_rows:,}  missing={r.missing.total_affected_rows}  '
          f'dups={r.duplicates.count}  outliers={r.outliers.count}  '
          f'imbalance={r.imbalance.imbalance_ratio:.2f}×')